In [ ]:
import pandas as pd
import numpy as np
from plotnine import *
from scipy.stats import *

In [ ]:
# Leer datos
datos = pd.read_csv("llegadas_restaurante.csv")

In [ ]:
# =====================================================
# Gráfico de barras para el tamaño de los grupos
# =====================================================
(ggplot(datos) +
    geom_bar(aes(x='factor(tamano)'), color="black", fill="yellow") +
    labs(title="Distribución del tamaño de grupos",
         x="Tamaño del grupo",
         y="Frecuencia") +
    theme_minimal()
)

In [ ]:
# =====================================================
# Convertir hora y calcular tiempo entre llegadas
# =====================================================
datos['hora'] = pd.to_datetime(datos['hora'], format='%H:%M')

# Ordenar por hora
datos = datos.sort_values('hora').reset_index(drop=True)

# Calcular tiempo entre llegadas en minutos
datos['tiempo_entre_llegadas'] = datos['hora'].diff().dt.total_seconds() / 60
datos = datos.dropna(subset=['tiempo_entre_llegadas']).reset_index(drop=True)

datos.head()

In [ ]:
# =====================================================
# Resumen estadístico
# =====================================================
print("Resumen estadístico de tiempos entre llegadas:")
print(datos['tiempo_entre_llegadas'].describe())
print(f"\nMedia: {datos['tiempo_entre_llegadas'].mean():.4f}")
print(f"Número de observaciones: {len(datos)}")
print(f"Desviación estándar: {datos['tiempo_entre_llegadas'].std():.4f}")

In [ ]:
# =====================================================
# Parámetros empíricos
# =====================================================
beta = datos['tiempo_entre_llegadas'].mean()   # media = beta (parámetro de escala)
n_obs = len(datos)

In [ ]:
cuantiles_teoricos = expon.ppf(np.linspace(0.01, 0.99, n_obs), scale=beta)

(ggplot() +
    geom_point(aes(x=cuantiles_teoricos,
                   y=sorted(datos['tiempo_entre_llegadas']))) +
    geom_abline(intercept=0, slope=1, color="red", linetype="dashed") +
    labs(title="Gráfico QQ - Tiempos entre llegadas vs Exponencial",
         x="Cuantiles teóricos (Exponencial)",
         y="Cuantiles observados") +
    theme_minimal()
)

In [ ]:
# =====================================================
# Histograma con curva exponencial superpuesta
# =====================================================
(ggplot(datos, aes(x='tiempo_entre_llegadas')) +
    geom_histogram(aes(y=after_stat('density')),
                   bins=30, color="black", fill="yellow", alpha=0.7) +
    stat_function(fun=lambda x: expon.pdf(x, scale=beta),
                  color="red", size=1.2) +
    labs(title="Histograma de tiempos entre llegadas",
         subtitle="Con curva teórica exponencial superpuesta",
         x="Tiempo entre llegadas (minutos)",
         y="Densidad") +
    theme_minimal()
)